In [ ]:
!pip install -q fastapi uvicorn[standard] python-multipart nest_asyncio

In [ ]:
%%writefile config.py

import torch

# -----------------------------
# Configuration Settings
# -----------------------------
BASE_MODEL_PATH = "C:/Users/khali/OneDrive/Bureau/machine learning Models/medical rag/RAG/Mistral-7B-Instruct-v0.2"
LORA_REPO_ID = "ysn-ir/mistral-medical-chat-lora-v1"

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
%%writefile schemas.py
from pydantic import BaseModel

class UserRequest(BaseModel):
    message: str
    max_tokens: int = 200
    temperature: float = 0.1

class BotResponse(BaseModel):
    response: str

In [ ]:
%%writefile model_service.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import config

class MedicalLLM:
    def __init__(self):
        print("⏳ Loading Tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL_PATH)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print("⏳ Loading Base Model (4-bit)...")
        self.base_model = AutoModelForCausalLM.from_pretrained(
            config.BASE_MODEL_PATH,
            torch_dtype=torch.float16,
            load_in_4bit=True,
            device_map="auto"
        )
        
        print(f"⏳ Loading LoRA Adapter: {config.LORA_REPO_ID}...")
        self.model = PeftModel.from_pretrained(
            self.base_model,
            config.LORA_REPO_ID,
            torch_dtype=torch.float16
        )
        self.model.eval()
        print("✅ Model Loaded Successfully!")

    def generate(self, user_message: str, max_tokens: int = 200, temperature: float = 0.1):
        # Apply the specific prompt engineering you requested
        if "patient" in user_message.lower() or "headache" in user_message.lower():
            prompt = (
                "You are a medical AI assistant. Provide general, safe advice only. "
                "Do not give dangerous instructions or mention extreme procedures. "
                "If the situation might be serious, instruct the patient to consult a licensed doctor immediately.\n"
                f"Patient: {user_message}\nDoctor:"
            )
        else:
            prompt = user_message

        inputs = self.tokenizer(prompt, return_tensors="pt").to(config.DEVICE)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=(temperature > 0),
                temperature=temperature if temperature > 0 else 1.0, # Avoid 0.0 error in some versions
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        full_response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Clean up response to remove the prompt part if desired
        # For now, we return the whole thing or split by "Doctor:" if you prefer only the answer
        return full_response

# Create a singleton instance to be imported
llm_engine = MedicalLLM()

In [ ]:
%%writefile main.py
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from fastapi.staticfiles import StaticFiles
import uvicorn
import os

from schemas import UserRequest, BotResponse
# Import the engine we created in the previous step
from model_service import llm_engine 

app = FastAPI(title="Mistral Medical Chatbot")

# Create templates directory if it doesn't exist
if not os.path.exists("templates"):
    os.makedirs("templates")

templates = Jinja2Templates(directory="templates")

@app.get("/", response_class=HTMLResponse)
async def read_root(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})

@app.post("/chat", response_model=BotResponse)
async def chat(request: UserRequest):
    print(f"📩 Received: {request.message}")
    response_text = llm_engine.generate(
        user_message=request.message,
        max_tokens=request.max_tokens,
        temperature=request.temperature
    )
    return BotResponse(response=response_text)

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>Medical Mistral Chat</title>
    <style>
        body { font-family: sans-serif; max-width: 800px; margin: 0 auto; padding: 20px; background-color: #f4f4f9; }
        .chat-container { height: 500px; overflow-y: scroll; border: 1px solid #ccc; padding: 20px; background: white; border-radius: 8px; margin-bottom: 20px;}
        .message { margin: 10px 0; padding: 10px; border-radius: 8px; max-width: 80%; }
        .user { background-color: #007bff; color: white; margin-left: auto; text-align: right; }
        .bot { background-color: #e9ecef; color: black; margin-right: auto; }
        .input-group { display: flex; gap: 10px; }
        input { flex-grow: 1; padding: 10px; border-radius: 4px; border: 1px solid #ccc; }
        button { padding: 10px 20px; background-color: #28a745; color: white; border: none; border-radius: 4px; cursor: pointer; }
        button:disabled { background-color: #ccc; }
        .loader { font-size: 12px; color: #666; font-style: italic; display: none; }
    </style>
</head>
<body>
    <h2>🏥 Local Medical Chatbot (Mistral + LoRA)</h2>
    <div class="chat-container" id="chatbox"></div>
    
    <div class="loader" id="loader">Doctor is thinking...</div>
    
    <div class="input-group">
        <input type="text" id="userInput" placeholder="Describe your symptoms..." onkeypress="handleEnter(event)">
        <button onclick="sendMessage()" id="sendBtn">Send</button>
    </div>

    <script>
        async function sendMessage() {
            const input = document.getElementById("userInput");
            const chatbox = document.getElementById("chatbox");
            const loader = document.getElementById("loader");
            const btn = document.getElementById("sendBtn");
            const text = input.value;

            if (!text) return;

            // Add User Message
            chatbox.innerHTML += `<div class="message user">${text}</div>`;
            input.value = "";
            loader.style.display = "block";
            btn.disabled = true;
            chatbox.scrollTop = chatbox.scrollHeight;

            try {
                const response = await fetch("/chat", {
                    method: "POST",
                    headers: { "Content-Type": "application/json" },
                    body: JSON.stringify({ message: text })
                });
                const data = await response.json();
                
                // Add Bot Message
                // Basic formatting for newlines
                const formattedResponse = data.response.replace(/\n/g, "<br>");
                chatbox.innerHTML += `<div class="message bot">${formattedResponse}</div>`;
            } catch (error) {
                chatbox.innerHTML += `<div class="message bot" style="color:red">Error connecting to model.</div>`;
            }

            loader.style.display = "none";
            btn.disabled = false;
            chatbox.scrollTop = chatbox.scrollHeight;
        }

        function handleEnter(e) {
            if (e.key === "Enter") sendMessage();
        }
    </script>
</body>
</html>

In [ ]:
%%writefile rag_service.py
import time
import uuid
import config
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

class RAGService:
    def __init__(self):
        print("📚 Loading Embedding Model (all-MiniLM-L6-v2)...")
        self.encoder = SentenceTransformer(config.EMBEDDING_MODEL_NAME)
        
        print("🌲 Connecting to Pinecone...")
        self.pc = Pinecone(api_key=config.PINECONE_API_KEY)
        
        # Check if index exists, create if not (Serverless spec for free tier)
        existing_indexes = [i.name for i in self.pc.list_indexes()]
        if config.PINECONE_INDEX_NAME not in existing_indexes:
            print(f"🌲 Creating new index: {config.PINECONE_INDEX_NAME}...")
            self.pc.create_index(
                name=config.PINECONE_INDEX_NAME,
                dimension=384, # Dimension for all-MiniLM-L6-v2
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1")
            )
            time.sleep(2) # Wait for initialization
            
        self.index = self.pc.Index(config.PINECONE_INDEX_NAME)
        print("✅ RAG Service Ready!")

    def chunk_text(self, text, chunk_size=500, overlap=50):
        """Simple helper to split text into chunks"""
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            chunks.append(chunk)
            start += (chunk_size - overlap)
        return chunks

    def ingest_file(self, filename: str, content: str):
        """Process text, embed it, and upload to Pinecone"""
        print(f"📄 Processing {filename}...")
        chunks = self.chunk_text(content)
        
        vectors = []
        for chunk in chunks:
            # Create embedding
            embedding = self.encoder.encode(chunk).tolist()
            # Create metadata
            metadata = {"filename": filename, "text": chunk}
            # Create vector ID
            vector_id = str(uuid.uuid4())
            
            vectors.append({"id": vector_id, "values": embedding, "metadata": metadata})
            
        # Upload in batches of 100
        batch_size = 100
        for i in range(0, len(vectors), batch_size):
            batch = vectors[i:i+batch_size]
            self.index.upsert(vectors=batch)
            
        return len(chunks)

    def search(self, query: str, top_k=3):
        """Search Pinecone for relevant context"""
        query_embedding = self.encoder.encode(query).tolist()
        
        results = self.index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True
        )
        
        contexts = [match['metadata']['text'] for match in results['matches']]
        return "\n\n".join(contexts)

# Singleton instance
rag_engine = RAGService()

In [ ]:
import uvicorn
import nest_asyncio
from main import app

# 1. Apply the patch to allow nested loops (just in case)
nest_asyncio.apply()

# 2. Configure the server
config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)

print("🚀 Starting Server in Notebook Mode...")
print("👉 Open this link in your browser: http://127.0.0.1:8000")

# 3. Run the server using the existing Jupyter loop
# We use 'await' because Jupyter cells support top-level await
await server.serve()

INFO:     Started server process [22004]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


🚀 Starting Server in Notebook Mode...
👉 Open this link in your browser: http://127.0.0.1:8000
INFO:     127.0.0.1:7353 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:7353 - "GET /static/css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:29240 - "GET /static/js/script.js HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:50788 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:50788 - "GET /static/css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:18591 - "GET /statsic/js/script.js HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:48391 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:48391 - "GET /static/css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:45254 - "GET /statsic/js/script.js HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:27346 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:27346 - "GET /static/css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:20432 - "GET /r%22C%3A/Users/khali/OneDrive/Bureau/medical HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:32819 - "GET / HTTP/1.1" 

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\fastapi\applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    raise exc
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\middleware

INFO:     127.0.0.1:47275 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:47275 - "GET /static/css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:43101 - "GET /static/js/script.js HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:62322 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:62322 - "GET /static/css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:41446 - "GET /static/js/script.js HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:8033 - "GET / HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\fastapi\applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    raise exc
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\middleware

INFO:     127.0.0.1:59472 - "GET / HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\fastapi\applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    raise exc
  File "c:\Users\khali\anaconda3\envs\medical-llm\lib\site-packages\starlette\middleware

INFO:     127.0.0.1:45603 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:45603 - "GET /css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:1173 - "GET /js/script.js HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:1173 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:1173 - "GET /css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:45603 - "GET /js/script.js HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:45603 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:45603 - "GET /css/style.css HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:1173 - "GET /js/script.js HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:45820 - "GET / HTTP/1.1" 200 OK
📩 Query: hello , i want to know what is the cause of cold feet , i am 20 years old and i have cold feet and ankles
🔎 Searching documents...
INFO:     127.0.0.1:18339 - "POST /chat HTTP/1.1" 200 OK
